In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

In [3]:
# 生成不平衡数据（正类10%，负类90%）
X, y = make_classification(
    n_samples=1000, n_features=2, n_redundant=0,
    weights=[0.1, 0.9], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"训练集正类: {sum(y_train==1)}，负类: {sum(y_train==0)}")

训练集正类: 626，负类: 74


In [4]:
# 评估函数
def evaluate(y_true, y_pred, name):
    print(f"{name:12s} | 准确率: {accuracy_score(y_true, y_pred):.3f} | "
          f"召回率: {recall_score(y_true, y_pred):.3f} | "
          f"F1: {f1_score(y_true, y_pred):.3f}")

In [10]:
# ===== （不处理） =====
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
evaluate(y_test, y_pred, "不处理")

不处理          | 准确率: 0.947 | 召回率: 0.993 | F1: 0.971


In [15]:
# ===== 欠采样 =====
sampler = RandomUnderSampler(random_state=42)
X_res, y_res = sampler.fit_resample(X_train, y_train)
clf_under = LogisticRegression(max_iter=1000).fit(X_res, y_res)
evaluate(y_test, clf_under.predict(X_test), "欠采样")

欠采样          | 准确率: 0.830 | 召回率: 0.818 | F1: 0.896


In [14]:
# ===== 过采样(SMOTE) =====
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
clf_over = LogisticRegression(max_iter=1000).fit(X_res, y_res)
evaluate(y_test, clf_over.predict(X_test), "SMOTE过采样")

SMOTE过采样     | 准确率: 0.847 | 召回率: 0.844 | F1: 0.908


In [17]:
# ===== 阈值移动1 =====
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred_thresh = (y_proba >= 0.15).astype(int)  # 阈值从0.5降到0.15
evaluate(y_test, y_pred_thresh, "阈值移动(0.15)")

阈值移动(0.15)   | 准确率: 0.907 | 召回率: 1.000 | F1: 0.951


In [20]:
# ===== 阈值移动2 =====
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred_thresh = (y_proba >= 0.05).astype(int)  # 阈值从0.15到0.05
evaluate(y_test, y_pred_thresh, "阈值移动(0.05)")

阈值移动(0.05)   | 准确率: 0.897 | 召回率: 1.000 | F1: 0.946


In [21]:
# ===== 阈值移动3 =====
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred_thresh = (y_proba >= 0.20).astype(int)  # 阈值从0.05降到0.20
evaluate(y_test, y_pred_thresh, "阈值移动(0.20)")

阈值移动(0.20)   | 准确率: 0.910 | 召回率: 1.000 | F1: 0.952


In [9]:
# ===== sklearn内置权重 =====
clf_weight = LogisticRegression(class_weight='balanced', max_iter=1000)
clf_weight.fit(X_train, y_train)
evaluate(y_test, clf_weight.predict(X_test), "内置权重")

内置权重         | 准确率: 0.843 | 召回率: 0.840 | F1: 0.906
